In [ ]:
import json
import os
import pickle
from typing import Tuple

import cv2
import numpy as np
import polars as pl
import scipy.optimize
import scipy.spatial
from numba import njit
from PIL import Image
from tqdm import tqdm
import imageio

from yagm.utils.geometric import batch_perspective_transform_2d
from ecg.utils.misc import get_scale_xy_from_homo_mat
from ecg.utils.viz import viz_img_keypoints

# ==========================================
# GIF VISUALIZATION SETTINGS & HELPERS
# ==========================================
# Speeds in milliseconds (ms)
HOLD_SPEED = 5000        # 2.0 seconds for key intro phases
FAST_SPEED = 1000         # 0.1 seconds (10 fps) for the homography loop
ROUND_HOLD_SPEED = 3000  # 1.5 seconds to compare end-of-round results
FINAL_HOLD = 5000        # 4.0 seconds to observe final results

# Define High-Contrast BGR Colors
COLOR_GREEN = (0, 255, 0)      # Existing Matches
COLOR_MAGENTA = (255, 0, 255)  # NEW Matches (Replaces Yellow)
COLOR_CYAN = (255, 255, 0)     # Connecting Lines (Replaces Yellow)
COLOR_BLUE = (255, 0, 0)       # Target points
COLOR_ORANGE = (0, 165, 255)   # Final Interpolated points
COLOR_RED = (0, 0, 255)

def create_gif_frame(image_bgr, text_title, sub_left=None, sub_right=None, target_width=1600, target_height=900):
    """
    Resizes and pads an image, adding a Main Title and optional Left/Right Subtitles above the image.
    No text is drawn over the actual image data.
    """
    h, w = image_bgr.shape[:2]
    
    # 1. Determine scale factor to fit target box
    scale = min(target_width / w, target_height / h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized_img = cv2.resize(image_bgr, (new_w, new_h))
    
    # 2. Define UI Heights (FIXED: Always reserve 50px for subtitles so frame sizes never change)
    title_height = 80
    subtitle_height = 50 
    total_header = title_height + subtitle_height
    
    # 3. Create Canvas
    final_canvas = np.zeros((target_height + total_header, target_width, 3), dtype=np.uint8)
    
    # 4. Paste Image
    y_offset = total_header + (target_height - new_h) // 2
    x_offset = (target_width - new_w) // 2
    final_canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_img
    
    # 5. Draw Main Title
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(final_canvas, text_title, (30, 50), font, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
    
    # 6. Draw Subtitles (if provided)
    if sub_left or sub_right:
        # Draw a subtle separator line between title and subtitles
        cv2.line(final_canvas, (30, title_height-10), (target_width-30, title_height-10), (100, 100, 100), 2)
        
        sub_font_scale = 0.9
        sub_color = (200, 200, 200) # Light grey for subtitles
        
        if sub_left:
            cv2.putText(final_canvas, sub_left, (x_offset, title_height + 35), font, sub_font_scale, sub_color, 2, cv2.LINE_AA)
        if sub_right:
            # Place right subtitle at the midpoint of the image width
            right_x = x_offset + (new_w // 2) + 20
            cv2.putText(final_canvas, sub_right, (right_x, title_height + 35), font, sub_font_scale, sub_color, 2, cv2.LINE_AA)
    
    return cv2.cvtColor(final_canvas, cv2.COLOR_BGR2RGB)


# ==========================================
# CORE ALGORITHM FUNCTIONS
# ==========================================

def check_points_inside_polygon(keypoints, polygon_coords):
    poly_array = np.array(polygon_coords, dtype=np.int32)
    mask = []
    for point in keypoints:
        pt_tuple = (float(point[0]), float(point[1]))
        result = cv2.pointPolygonTest(poly_array, pt_tuple, measureDist=False)
        mask.append(result >= 0)
    return np.array(mask, dtype=bool)


@njit
def greedy_match(pred_xy, pred_score, gt_xy, threshold, cost_gating = None):
    del cost_gating
    P = pred_xy.shape[0]
    G = gt_xy.shape[0]
    order = np.argsort(-pred_score)
    sorted_pred_xy = pred_xy[order]
    gt_used = np.zeros(G, dtype=np.bool_)
    pred_used_sorted_idx = np.zeros(P, dtype=np.bool_) 
    max_matches = min(P, G)
    temp_matches = np.zeros((max_matches, 2), dtype=np.int64)
    match_count = 0

    for i in range(P):
        if G == 0: break
        min_dist = 1e12
        min_j = -1
        for j in range(G):
            if gt_used[j]: continue
            dx = gt_xy[j, 0] - sorted_pred_xy[i, 0]
            dy = gt_xy[j, 1] - sorted_pred_xy[i, 1]
            d = (dx * dx + dy * dy) ** 0.5
            if d < min_dist:
                min_dist = d
                min_j = j
        if min_j >= 0 and min_dist <= threshold:
            gt_used[min_j] = True
            pred_used_sorted_idx[i] = True
            temp_matches[match_count, 0] = order[i]
            temp_matches[match_count, 1] = min_j
            match_count += 1

    matches = temp_matches[:match_count]
    all_gt_indices = np.arange(G)
    unmatched_gts = all_gt_indices[~gt_used]
    pred_used_original = np.zeros(P, dtype=np.bool_)
    for i in range(P):
        if pred_used_sorted_idx[i]:
            pred_used_original[order[i]] = True

    all_pred_indices = np.arange(P)
    unmatched_preds = all_pred_indices[~pred_used_original]
    return matches, unmatched_preds, unmatched_gts


EPSILON = 1e-5
POS_INF = np.finfo(np.float32).max


def _linear_assignment_hungarian(cost_matrix: np.ndarray, max_cost: float, cost_gating=None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    cost_matrix = cost_matrix.copy()
    if cost_gating is not None:
        cost_matrix[cost_matrix > cost_gating] = 999999

    row_ind, col_ind = scipy.optimize.linear_sum_assignment(cost_matrix)
    matches = []
    for ia, ib in zip(row_ind, col_ind):
        if cost_matrix[ia, ib] <= max_cost:
            matches.append([ia, ib])

    matches = np.asarray(matches)
    if len(matches) > 0:
        unmatched_a = np.asarray([idx for idx in range(cost_matrix.shape[0]) if idx not in matches[:, 0]])
        unmatched_b = np.asarray([idx for idx in range(cost_matrix.shape[1]) if idx not in matches[:, 1]])
    else:
        unmatched_a = np.arange(cost_matrix.shape[0])
        unmatched_b = np.arange(cost_matrix.shape[1])
        matches = np.empty((0, 2), dtype=int)
    return matches, unmatched_a, unmatched_b


def hungarian_match(pred_xy, pred_score, gt_xy, threshold, cost_gating = None):
    P = pred_xy.shape[0]
    G = gt_xy.shape[0]
    if P == 0: return np.empty((0, 2), dtype=int), np.empty(0, dtype=int), np.arange(G)
    if G == 0: return np.empty((0, 2), dtype=int), np.arange(P), np.empty(0, dtype=int)

    cost_matrix = scipy.spatial.distance.cdist(pred_xy, gt_xy, metric="euclidean")
    matches, unmatched_preds, unmatched_gts = _linear_assignment_hungarian(cost_matrix, threshold, cost_gating = cost_gating)
    return matches, unmatched_preds, unmatched_gts


def visualize_matches(image_rgb, matches, unmatched_preds, unmatched_gts, pred_xy, gt_xy):
    if image_rgb is None:
        canvas = np.zeros((1700, 2200, 3), dtype=np.uint8)
    else:
        canvas = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    COLOR_MATCH = (0, 255, 0)
    COLOR_UNMATCH_PRED = (0, 0, 255)
    COLOR_UNMATCH_GT = (255, 0, 0)

    if gt_xy is not None:
        for idx in unmatched_gts:
            pt = tuple(gt_xy[idx].astype(int))
            cv2.circle(canvas, pt, radius=8, color=COLOR_UNMATCH_GT, thickness=2)

    for idx in unmatched_preds:
        pt = tuple(pred_xy[idx].astype(int))
        cv2.circle(canvas, pt, radius=5, color=COLOR_UNMATCH_PRED, thickness=-1)

    for pred_idx, gt_idx in matches:
        pt_pred = tuple(pred_xy[pred_idx].astype(int))
        if gt_xy is not None:
            pt_gt = tuple(gt_xy[gt_idx].astype(int))
            cv2.line(canvas, pt_pred, pt_gt, (0, 0, 255), thickness=2)
            cv2.circle(canvas, pt_gt, radius=8, color=COLOR_MATCH, thickness=2)
        cv2.circle(canvas, pt_pred, radius=4, color=(255, 0, 0), thickness=-1)

    return canvas


def load_img(sample):
    sample_id = sample["id"]
    type_id = sample["type_id"]
    rot_code = sample["rot_code"]
    img_path = os.path.join("/home/dangnh36/datasets/ecg/raw/train/", str(sample_id), f"{sample_id}-{type_id:04d}.png")
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if rot_code is not None:
        img = cv2.rotate(img, int(rot_code))
    return img


def filter_matches_largest_cca(matched_ori_gt_idxs, matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs):
    matched_mask = np.zeros((NUM_GRID_KPT, ), dtype='uint8')
    matched_mask[matched_ori_gt_idxs] = 1
    matched_mask = matched_mask.reshape(43, 55)
    num_labels, labels = cv2.connectedComponents(matched_mask, connectivity=4)
    seg_counts = np.bincount(labels.flatten())
    if len(seg_counts) > 1:
        seg_counts[0] = -999 
        largest_label = np.argmax(seg_counts)
        labels_flat = labels.flatten()
        is_in_largest_cc = labels_flat[matched_ori_gt_idxs] == largest_label
        prune_matched_idxs = matched_idxs[~is_in_largest_cc]
        matched_idxs = matched_idxs[is_in_largest_cc]
    else:
        prune_matched_idxs = matched_idxs
        matched_idxs = np.empty((0, 2), dtype=int)

    if len(prune_matched_idxs) > 0:
        unmatched_pred_idxs = np.concatenate([unmatched_pred_idxs, prune_matched_idxs[:, 0]], axis=0).astype(int)
        unmatched_gt_idxs = np.concatenate([unmatched_gt_idxs, prune_matched_idxs[:, 1]], axis=0).astype(int)
    return matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs

def filter_matches_min_connections(matched_ori_gt_idxs, matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs, min_connections=3):
    matched_mask = np.zeros((NUM_GRID_KPT, ), dtype='float32') 
    matched_mask[matched_ori_gt_idxs] = 1
    matched_mask = matched_mask.reshape(43, 55)
    kernel = np.ones((3, 3), dtype=np.float32)
    kernel[1, 1] = 0
    neighbor_counts = cv2.filter2D(matched_mask, -1, kernel, borderType=cv2.BORDER_CONSTANT)
    is_valid_mask = (matched_mask == 1) & (neighbor_counts >= min_connections)
    is_valid_flat = is_valid_mask.flatten()
    keep_indices = is_valid_flat[matched_ori_gt_idxs]
    prune_matched_idxs = matched_idxs[~keep_indices]
    matched_idxs = matched_idxs[keep_indices]
    if len(prune_matched_idxs) > 0:
        unmatched_pred_idxs = np.concatenate([unmatched_pred_idxs, prune_matched_idxs[:, 0]]).astype(int)
        unmatched_gt_idxs = np.concatenate([unmatched_gt_idxs, prune_matched_idxs[:, 1]]).astype(int)
    return matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs

def filter_matches_connected_to_matched_mask(base_matched_ori_gt_idxs, matched_ori_gt_idxs, matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs):
    base_mask = np.zeros((NUM_GRID_KPT, ), dtype='uint8')
    if len(base_matched_ori_gt_idxs) > 0: base_mask[base_matched_ori_gt_idxs] = 1
    base_mask = base_mask.reshape(43, 55)
    current_mask = np.zeros((NUM_GRID_KPT, ), dtype='uint8')
    if len(matched_ori_gt_idxs) > 0: current_mask[matched_ori_gt_idxs] = 1
    current_mask = current_mask.reshape(43, 55)
    union_mask = cv2.bitwise_or(current_mask, base_mask)
    num_labels, labels = cv2.connectedComponents(union_mask, connectivity=4)
    anchored_labels = np.unique(labels[base_mask == 1])
    anchored_labels = anchored_labels[anchored_labels != 0] 
    if len(anchored_labels) > 0:
        labels_flat = labels.flatten()
        match_labels = labels_flat[matched_ori_gt_idxs]
        is_anchored = np.isin(match_labels, anchored_labels)
        prune_matched_idxs = matched_idxs[~is_anchored]
        matched_idxs = matched_idxs[is_anchored]
    else:
        prune_matched_idxs = matched_idxs
        matched_idxs = np.empty((0, 2), dtype=int)
    if len(prune_matched_idxs) > 0:
        unmatched_pred_idxs = np.concatenate([unmatched_pred_idxs, prune_matched_idxs[:, 0]]).astype(int)
        unmatched_gt_idxs = np.concatenate([unmatched_gt_idxs, prune_matched_idxs[:, 1]]).astype(int)
    return matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs


# ==========================================
# MAIN EXECUTION & DATA LOADING
# ==========================================
FOLD = 3
PREDICTION_PATH = f"/home/dangnh36/datasets/ecg/processed/pseudo_label/round1_finetune_dsnt/fold{FOLD}_predictions.pkl"
PREDICTION_DIR = os.path.dirname(PREDICTION_PATH)

with open(PREDICTION_PATH, "rb") as f:
    data = pickle.load(f)

MAIN_KPT_METHOD = "dsnt"
gt_df = data["gt_df"]
all_main_kpt_pred = data["main_kpt_preds"][MAIN_KPT_METHOD]
all_grid_kpt_pred = data["grid_kpt_pred"]

with open("/home/dangnh36/datasets/ecg/processed/reference_keypoints.json", "r") as f:
    ref_kpts = json.load(f)
ref_kpt_xys = np.array(list(ref_kpts.values()))

STD_REF_GRID_KPT_XYS = ref_kpt_xys[:2365]
_min_x, _max_x = STD_REF_GRID_KPT_XYS[:, 0].min(), STD_REF_GRID_KPT_XYS[:, 0].max()
_min_y, _max_y = STD_REF_GRID_KPT_XYS[:, 1].min(), STD_REF_GRID_KPT_XYS[:, 1].max()

MARGIN = 30
STD_REF_SAFE_POLYGON = np.array([
    [_min_x - MARGIN, _min_y - MARGIN], [_max_x + MARGIN, _min_y - MARGIN],
    [_max_x + MARGIN, _max_y + MARGIN], [_min_x - MARGIN, _max_y + MARGIN],
], dtype=np.float32)

REF_IMG = load_img(gt_df[0].to_dicts()[0])
NUM_GRID_KPT = 2365
SKIP_MATCHED_SIZE = False
FILTER_OUTSIDE = False
CONF_THRES = 0.05
MATCH_FUNCTION = hungarian_match
DIST_THRES1 = 15 
DIST_THRES2 = 8
COST_GATING1 = 28.3
COST_GATING2 = 28.3 

VIZ = True
VERBOSE = True

all_final_grid_kpt = np.zeros((len(gt_df), 2365, 2), dtype=np.float32)

# ==========================================
# ALGORITHM LOOP & GIF RECORDING
# ==========================================
for i in tqdm(range(len(gt_df))):
    final_grid_kpt = np.zeros_like(STD_REF_GRID_KPT_XYS)
    sample = gt_df[i].to_dicts()[0]
    img = None
    
    DEBUG_IDX = 1492
    if i != DEBUG_IDX:
        continue

    # Initialize Memory-Efficient GIF arrays
    gif_frames = []
    frame_durations = []

    grid_kpt = all_grid_kpt_pred[i]

    if len(grid_kpt) > NUM_GRID_KPT and CONF_THRES is not None:
        if CONF_THRES == 'top':
            sorted_idxs = np.argsort(-grid_kpt[:, 2])
            grid_kpt = grid_kpt[sorted_idxs[:NUM_GRID_KPT]]
        else:
            grid_kpt = grid_kpt[grid_kpt[:, 2] > CONF_THRES]

    if sample["H"] is None:
        ori_scale_x = ori_scale_y = 1.0
        ref_grid_kpt = grid_kpt.copy()[:, :2]
    else:
        to_ref_H = np.array(eval(sample["H"]))
        from_ref_H = np.linalg.pinv(to_ref_H)
        ori_scale_x, ori_scale_y = get_scale_xy_from_homo_mat(from_ref_H)
        ref_grid_kpt = batch_perspective_transform_2d(grid_kpt[None, :, :2], to_ref_H[None])[0]

    matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs = MATCH_FUNCTION(
        ref_grid_kpt, grid_kpt[:, 2], STD_REF_GRID_KPT_XYS, threshold=DIST_THRES1, cost_gating = COST_GATING1
    )

    matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs = filter_matches_largest_cca(matched_idxs[:, 1], matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs)
    matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs = filter_matches_min_connections(matched_idxs[:, 1], matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs, min_connections=4)
    
    # ===== GIF STEP 1: INITIAL PHASE =====
    # ===== GIF STEP 1: INITIAL PHASE =====
    if VIZ:
        img = load_img(sample)
        # 1a. Raw Predictions Plotted
        raw_pred_viz = img.copy()
        raw_pred_viz = cv2.cvtColor(raw_pred_viz, cv2.COLOR_RGB2BGR)
        for pt in grid_kpt[:, :2].astype(int):
            cv2.circle(raw_pred_viz, tuple(pt), 4, COLOR_BLUE, -1) 
            
        frame_step0 = create_gif_frame(raw_pred_viz, "STAGE 0: Raw Keypoint Predictions")
        gif_frames.append(frame_step0)
        frame_durations.append(HOLD_SPEED)

        # 1b. Phase 1 Matches
        viz_pred = visualize_matches(img, matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs, grid_kpt[:, :2], None)
        viz_ref = visualize_matches(REF_IMG, matched_idxs, unmatched_pred_idxs, unmatched_gt_idxs, ref_grid_kpt, STD_REF_GRID_KPT_XYS)
        
        # Stack them cleanly (no text on them)
        h_ref = viz_ref.shape[0]
        scale_factor = h_ref / viz_pred.shape[0]
        viz_pred_resized = cv2.resize(viz_pred, (int(viz_pred.shape[1] * scale_factor), h_ref))
        combined_phase1 = np.hstack([viz_ref, viz_pred_resized])
        
        # Pass the descriptions to the new subtitle arguments
        main_title = f"STAGE 1: Phase 1 Matches (Hungarian + CCA) - Matched {len(matched_idxs)} points (blue)"
        txt_left = "Hungarian results in Reference Space"
        txt_right = "Matched/Unmatched (blue/red) in Current Space"
        
        frame_step1 = create_gif_frame(combined_phase1, main_title, sub_left=txt_left, sub_right=txt_right)
        gif_frames.append(frame_step1)
        frame_durations.append(HOLD_SPEED)

    matches = np.concatenate([matched_idxs, grid_kpt[matched_idxs[:, 0], :2], STD_REF_GRID_KPT_XYS[matched_idxs[:, 1]]], axis=1)
    unmatched_gts = np.concatenate([STD_REF_GRID_KPT_XYS[unmatched_gt_idxs], unmatched_gt_idxs[:, None]], axis=1) if len(unmatched_gt_idxs) > 0 else np.empty((0, 3), dtype="float32")
    unmatched_preds = np.concatenate([grid_kpt[unmatched_pred_idxs], unmatched_pred_idxs[:, None]], axis=1) if len(unmatched_pred_idxs) > 0 else np.empty((0, 3), dtype="float32")

    if len(matches):
        final_grid_kpt[matches[:, 1].astype(np.uint32)] = matches[:, 2:4]

    # ========== INTERATIVELY HANDLE UNMATCHED GT =========
    cur_round = 0
    all_matched_ori_gt_idxs2 = []
    unmatched2_ori_gt_idxs = []
    
    while True:
        if len(unmatched_gts) == 0: break
        cur_round += 1
        interpolated_xys = []
        
        for j, ref_gt_point in enumerate(unmatched_gts):
            cur_gt_xy = ref_gt_point[:2]
            dists = cur_gt_xy[None] - matches[:, 4:6]
            dists = (dists[:, 0] ** 2 + dists[:, 1] ** 2) ** 0.5
            sort_idxs = np.argsort(dists)
            nearest_idxs = sort_idxs[:24]
            nearest_matches = matches[nearest_idxs]

            src_pts = nearest_matches[:, 4:6]
            dst_pts = nearest_matches[:, 2:4]
            local_H, mask = cv2.findHomography(src_pts, dst_pts, cv2.USAC_MAGSAC, ransacReprojThreshold=8.0, maxIters=5000, confidence=0.9999)
            interpolated_pred_xy = cv2.perspectiveTransform(cur_gt_xy.reshape(1, 1, 2), local_H)[0, 0]
            interpolated_xys.append(interpolated_pred_xy)

            # ===== GIF STEP 2a: HOMOGRAPHY ANIMATION (ROUND 1 ONLY) =====
            # Capture the first 10 interpolations of ONLY round 1
            if VIZ and cur_round == 1 and j < 10: 
                vis_ref = cv2.cvtColor(REF_IMG.copy(), cv2.COLOR_RGB2BGR)
                vis_pred = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2BGR)

                center_ref = tuple(cur_gt_xy.astype(int))
                center_pred = tuple(interpolated_pred_xy.astype(int))

                for pt in src_pts.astype(int):
                    # Changed to CYAN for geometry lines
                    cv2.line(vis_ref, center_ref, tuple(pt), COLOR_CYAN, 1)
                    cv2.circle(vis_ref, tuple(pt), 5, COLOR_GREEN, -1)
                cv2.circle(vis_ref, center_ref, 10, COLOR_BLUE, -1) 

                for pt in dst_pts.astype(int):
                    # Changed to CYAN for geometry lines
                    cv2.line(vis_pred, center_pred, tuple(pt), COLOR_CYAN, 1)
                    cv2.circle(vis_pred, tuple(pt), 5, COLOR_GREEN, -1)
                cv2.circle(vis_pred, center_pred, 10, COLOR_BLUE, -1) 

                h_ref = vis_ref.shape[0]
                scale_factor = h_ref / vis_pred.shape[0]
                vis_pred_resized = cv2.resize(vis_pred, (int(vis_pred.shape[1] * scale_factor), h_ref))
                combined_view = np.hstack([vis_ref, vis_pred_resized])
                
                # Append to GIF at FAST speed
                frame_step2 = create_gif_frame(combined_view, f"STAGE 2: Interpolation using local H | Round 1, point {j+1}/{len(unmatched_gts)}")
                gif_frames.append(frame_step2)
                frame_durations.append(FAST_SPEED)

        interpolated_xys = np.array(interpolated_xys)

        thres2 = DIST_THRES2 * ((ori_scale_x**2 + ori_scale_y**2) ** 0.5) / (2**0.5)
        cost_gating2 = COST_GATING2 * ((ori_scale_x**2 + ori_scale_y**2) ** 0.5) / (2**0.5) if COST_GATING2 is not None else None
        
        matched_idxs2, unmatched_pred_idxs2, unmatched_gt_idxs2 = MATCH_FUNCTION(
            unmatched_preds[:, :2], unmatched_preds[:, 2], interpolated_xys, threshold=thres2, cost_gating = cost_gating2
        )
        
        if len(matched_idxs2) > 0:
            matched_idxs2, unmatched_pred_idxs2, unmatched_gt_idxs2 = filter_matches_connected_to_matched_mask(
                matches[:, 1].astype(int), unmatched_gts[matched_idxs2[:, 1], 2].astype(int), 
                matched_idxs2, unmatched_pred_idxs2, unmatched_gt_idxs2)

        if len(matched_idxs2) > 0:
            matched_ori_gt_idxs2 = unmatched_gts[matched_idxs2[:, 1], 2].astype(int)
            matched_ori_pred_idxs2 = unmatched_preds[matched_idxs2[:, 0], 3].astype(int)
            final_grid_kpt[matched_ori_gt_idxs2] = unmatched_preds[matched_idxs2[:, 0], :2]
            
            new_matches = np.concatenate([
                matched_ori_pred_idxs2[:, None], matched_ori_gt_idxs2[:, None],
                unmatched_preds[matched_idxs2[:, 0], :2], unmatched_gts[matched_idxs2[:, 1], :2],
            ], axis=1)
            matches = np.concatenate([matches, new_matches], axis=0)
        else:
            matched_ori_gt_idxs2 = []
            matched_ori_pred_idxs2 = []
            
        # ===== GIF STEP 2b: ROUND COMPLETE SUMMARY (ALL ROUNDS) =====
        if VIZ and len(matched_ori_gt_idxs2) > 0:
            round_viz = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2BGR)
            # Draw all matches found SO FAR in GREEN
            for pred_idx, gt_idx, px, py, gx, gy in matches:
                cv2.circle(round_viz, (int(px), int(py)), 5, COLOR_GREEN, -1)
            
            # Highlight the NEW matches from this round in HIGH-CONTRAST MAGENTA
            for px, py in unmatched_preds[matched_idxs2[:, 0], :2]:
                cv2.circle(round_viz, (int(px), int(py)), 6, COLOR_MAGENTA, -1) 
            
            frame_round_complete = create_gif_frame(round_viz, f"STAGE 2, round {cur_round} Hungarian matching: +{len(matched_ori_gt_idxs2)} predicted points matched")
            gif_frames.append(frame_round_complete)
            frame_durations.append(ROUND_HOLD_SPEED) 

        if len(unmatched_gt_idxs2) > 0:
            unmatched2_ori_gt_idxs = unmatched_gts[unmatched_gt_idxs2, 2].astype(int)
            unmatched_gts = unmatched_gts[unmatched_gt_idxs2]
        else:
            unmatched2_ori_gt_idxs = []
            unmatched_gts = np.empty((0, 3), dtype = 'float32')
            
        if len(unmatched_pred_idxs2) > 0:
            unmatched_preds = unmatched_preds[unmatched_pred_idxs2]
        else:
            unmatched_preds = np.empty((0, 3), dtype = 'float32')

        all_matched_ori_gt_idxs2.extend(matched_ori_gt_idxs2)

        if len(matched_ori_gt_idxs2) == 0:
            break

    if len(unmatched2_ori_gt_idxs) > 0:
        final_grid_kpt[unmatched2_ori_gt_idxs] = interpolated_xys[unmatched_gt_idxs2]

    # ===== GIF STEP 3: FINAL INTERPOLATED RESULT =====
    if VIZ:
        final_viz = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2BGR)
        for idx in range(2365):
            pt = tuple(final_grid_kpt[idx].astype(int))
            # Orange for interpolated, Green for matched
            color = COLOR_RED if idx in unmatched2_ori_gt_idxs else COLOR_GREEN
            cv2.circle(final_viz, pt, 5, color, -1)

        final_frame = create_gif_frame(final_viz, f"Final Output. Matched: {len(matches)} | Interpolated: {len(unmatched2_ori_gt_idxs)} (red)")
        gif_frames.append(final_frame)
        frame_durations.append(FINAL_HOLD)
            
        # ==========================================
        # EFFICIENT GIF EXPORT
        # ==========================================
        output_gif_path = os.path.join(PREDICTION_DIR, f"ecg_matching_algorithm_{DEBUG_IDX}.gif")
        
        imageio.mimsave(output_gif_path, gif_frames, duration=frame_durations, loop=0)
        print(f"✅ SUCCESS: Efficient Variable-Speed GIF saved to {output_gif_path}")

    all_final_grid_kpt[i] = final_grid_kpt

print("FINAL", all_final_grid_kpt.shape, all_main_kpt_pred.shape)
final_pred = np.concatenate([all_final_grid_kpt, all_main_kpt_pred], axis=1)
print(final_pred.shape)

In [ ]:
SAVE_PATH = os.path.join(PREDICTION_DIR, f'fold{FOLD}_refined.npy')
np.save(SAVE_PATH, final_pred)
print('Done!\n\n\n')

!ls -la {PREDICTION_DIR}

In [ ]:
fold_arrs = []
fold_dfs = []
for fold_idx in range(5):
    npy_path = os.path.join(PREDICTION_DIR, f'fold{fold_idx}_refined.npy')
    fold_arrs.append(np.load(npy_path))

    with open(os.path.join(PREDICTION_DIR, f'fold{fold_idx}_predictions.pkl'), 'rb') as f:
        d = pickle.load(f)
    fold_dfs.append(d['gt_df'].with_columns(pl.lit(fold_idx, dtype = pl.UInt16).alias('fold')))
    
ret = np.concatenate(fold_arrs, axis = 0)
print(ret.shape)
df = pl.concat(fold_dfs)
assert sorted(df['index'].to_list()) == list(range(len(df)))
print(df.shape)
display(df)

concat_idxs = df['index'].to_numpy()
ori_order_ret = np.zeros_like(ret)
ori_order_ret[concat_idxs] = ret
print(ori_order_ret.shape)
ori_order_ret

In [ ]:
np.save(os.path.join(PREDICTION_DIR, 'keypoints_by_round1.npy'), ori_order_ret)

## VISUALIZE + DEBUG

In [ ]:
gt_df = pl.read_csv('/home/dangnh36/datasets/ecg/processed/keypoints_by_model.csv')
gt_df

In [ ]:
all_kpts = np.load(os.path.join(PREDICTION_DIR, 'keypoints_by_round1.npy'))
all_kpts.shape

In [ ]:
OUTSIDE:
3136 2006667321 6 None (3000, 4000, 3)


WRONG:
7473 63496229 5 None (3024, 4032, 3)

8772 2512496238 10 None (3024, 4032, 3)

5880 3446730127 5 None (3024, 4032, 3)

2541 2219724680 5 None (3024, 4032, 3)

In [ ]:
import random

# select_idxs = random.sample(list(range(len(gt_df))), k = 50)
select_idxs = [7108]
# select_idxs = [7473]
for i in select_idxs:
    sample = gt_df[i].to_dicts()[0]
    # print(sample['id'])
    rgb = load_img(sample)
    print(i, sample['id'], sample['type_id'], sample['rot_code'], rgb.shape)
    kpts = all_kpts[i]
    
    viz_img_keypoints(
        rgb,
        kpt_xys = kpts,
        kpt_classes = [0] * 2365 + list(range(57)),
        save_path=None,
        figsize=(12, 12),
        point_size=10,
        alpha=0.9,
        show_legend=False,
    )
    print('-------------------\n\n\n')

### Re-estimate H and create proper CSV file

In [ ]:
with open(
    "/home/dangnh36/datasets/ecg/processed/reference_keypoints.json", "r"
) as f:
    ref_kpts = json.load(f)
ref_kpt_xys = np.array(list(ref_kpts.values()))
ref_kpt_names = list(ref_kpts.keys())
len(ref_kpt_names), ref_kpt_xys.shape

In [ ]:
gt_df = pl.read_csv('/home/dangnh36/datasets/ecg/processed/keypoints_by_model.csv')
gt_df

In [ ]:
all_kpts = np.load(os.path.join(PREDICTION_DIR, 'keypoints_by_round1.npy'))
all_kpts.shape

In [ ]:
# "ransac": {
#         "enable": True,
#         "estimator": "poselib",  # ??? since imcui did not use this kwarg
#         "geometry": "homography",
#         "method": "cv2_USAC_MAGSAC",
#         "reproj_threshold": 2,
#         "confidence": 0.9999,
#         "max_iter": 100000,
# }


all_final_kpts = []
all_final_rows = []
for i, row in tqdm(enumerate(gt_df.iter_rows(named = True)), total = len(gt_df)):
    type_id = row['type_id']
    if type_id == 1:
        # identity, use reference points instead of model's predictions
        final_kpts = ref_kpt_xys.copy()
        H = None
    else:
        final_kpts = all_kpts[i]
        # M: transformation matrix from current image TO REF image (to_ref_H)
        H, mask = cv2.findHomography(
            final_kpts,
            ref_kpt_xys,
            method=cv2.USAC_MAGSAC,
            ransacReprojThreshold=2,
            confidence=0.9999,
            maxIters=100_000,
        )
    new_row = {'id': row['id'], 'type_id': int(row['type_id']), 'rot_code': int(row['rot_code']) if row['rot_code'] is not None else None,
               'H': str(H.tolist()) if H is not None else H,
                'keypoints': str({kpt_name: kpt_xy for kpt_name, kpt_xy in zip(ref_kpt_names, final_kpts)})}
    all_final_kpts.append(final_kpts)
    all_final_rows.append(new_row)
    
all_final_kpts = np.array(all_final_kpts)
all_final_kpts.shape, len(all_final_rows)

In [ ]:
df = pl.DataFrame(all_final_rows)
df

In [ ]:
df.write_csv('/home/dangnh36/datasets/ecg/processed/keypoints_by_round1.csv')

In [ ]:
np.save('/home/dangnh36/datasets/ecg/processed/keypoints_by_round1.npy', all_final_kpts)

### Re-check the computed Homography matrix H

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import cv2
import ast

# 1. PARSING HELPER
def parse_h_matrix(h_val):
    """
    Parses the homography matrix string into a numpy array.
    Returns Identity matrix if H is null/None.
    """
    if h_val is None:
        return np.eye(3)
    
    try:
        # If it's already a numpy array (rare, but possible depending on loading)
        if isinstance(h_val, np.ndarray):
            return h_val
        # Parse string "[...]"
        return np.array(ast.literal_eval(h_val))
    except Exception as e:
        # Fallback to identity if parsing fails
        return np.eye(3)

def load_image(image_id, type_id, rot_code):
    img_path = os.path.join('/home/dangnh36/datasets/ecg/raw/train/', str(image_id), f"{image_id}-{type_id:04d}.png" )
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # correct the orientation
    if rot_code is not None:
        assert rot_code == int(rot_code)
        rot_code = int(rot_code)
        img = cv2.rotate(img, rot_code)    
    return img

# 2. VISUALIZATION LOGIC
def visualize_grid(df: pl.DataFrame, target_id: int):
    # Filter for the specific ID
    # Polars uses .filter(pl.col(...))
    subset = df.filter(pl.col("id") == target_id)
    
    if subset.is_empty():
        print(f"No data found for ID: {target_id}")
        return

    # Find reference row (type_id == 1) to determine target shape
    ref_row = subset.filter(pl.col("type_id") == 1)
    
    target_w, target_h = (300, 300) # Default
    if not ref_row.is_empty():
        # Load the actual reference image to get its shape
        # accessing first item of the column using .item()
        rid = ref_row["id"].item(0)
        rtype = ref_row["type_id"].item(0)
        rrotcode = ref_row['rot_code'].item(0)
        ref_img = load_image(rid, rtype, rrotcode)
        target_h, target_w = ref_img.shape[:2]

    # Select top 9 rows for the 3x3 grid
    grid_data = subset.head(9)

    # Setup Plot
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes = axes.flatten()

    # Iterate using iter_rows(named=True) for dictionary-like access
    for i, ax in enumerate(axes):
        if i < len(grid_data):
            # Extract row data
            row = grid_data.row(i, named=True)
            
            # Load Source Image
            src_img = load_image(row['id'], row['type_id'], row['rot_code'])
            
            # Parse H Matrix
            H = parse_h_matrix(row['H'])
            
            # Warp
            # Note: Warp output size is (width, height)
            warped_img = cv2.warpPerspective(src_img, H, (target_w, target_h))
            
            # Display
            ax.imshow(cv2.cvtColor(warped_img, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Type: {row['type_id']}")
            ax.axis('off')
        else:
            # Hide empty subplots
            ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
df = pl.read_csv('/home/dangnh36/datasets/ecg/processed/keypoints_by_round1.csv')
df

In [ ]:
df.filter(~pl.col('rot_code').is_null())

In [ ]:
target_ids = df['id'].unique().sample(20).to_list()

for target_id in target_ids:
    print(f'================= {target_id} =================')
    visualize_grid(df, target_id)